# Diabetes Health Indicators: Binary Classification EDA

This notebook is reconstructed from the annotated PDF report.

It converts the original `Diabetes_012` outcome into a binary target and performs a structured exploratory data analysis (EDA). It covers data quality, class balance, feature distributions, feature-target relationships, correlation, multicollinearity, and a simple baseline model.

> **Important target definition:** `Diabetes_binary = 1` when `Diabetes_012` is either `1` (prediabetes) or `2` (diabetes). Only original value `0` (non-diabetes) is assigned `Diabetes_binary = 0`.

## Notebook Objectives

- Load and inspect the diabetes health indicators dataset.
- Convert the original three-class target into a binary target:
  - **Negative (`0`)**: Non-diabetes only.
  - **Positive (`1`)**: Prediabetes or diabetes.
- Assess data quality and class balance.
- Export the cleaned binary dataset as a CSV file.
- Explore categorical-like and continuous-ish feature distributions.
- Examine feature relationships with the binary target.
- Check correlation and potential multicollinearity.
- Build a simple Random Forest baseline for exploratory comparison.

## AWS SageMaker Conversion Notes

This version preserves the EDA methodology, binary target definition, cleaning logic, visualizations, correlation analysis, multicollinearity check, and Random Forest baseline.

Only environment-specific I/O is changed:

- **Input:** The source dataset is retrieved from the supplied `DATA_SOURCE_URL` Kaggle public dataset page.
- **Output:** The cleaned binary dataset is written locally and uploaded to Amazon S3.
- **AWS SDK compatibility:** AWS access uses `boto3.Session()` directly.
- **AWS destination:** `s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv`

Run the notebook from top to bottom because later sections depend on variables created earlier.

## Table of Contents

1. Setup & Data Loading
2. Convert to Binary Target
3. Basic Data Quality Checks
3A. Export Cleaned Binary Dataset
4. Class Balance Visualization
5. Identify Numeric vs Categorical-like Features
6. Univariate Analysis: Histograms for Continuous-ish Features
7. Feature vs Target: Categorical-like Features
8. Feature vs Target: Continuous-ish Features (Boxplots)
9. Correlation Analysis
10. Multicollinearity Check (Highly Correlated Pairs)
11. Optional: Simple Baseline Model & Feature Importance
12. Short Text Summary Helper (Optional)

## 1. Setup & Data Loading

**Purpose:** Imports the required libraries, configures the SageMaker/AWS environment, retrieves the CSV selected by the supplied `DATA_SOURCE_URL`, and performs the same initial inspection of shape, sample records, data types, and the original target distribution.

**How to interpret this section:** Confirm that the Kaggle source file loads successfully in SageMaker and that `Diabetes_012` contains the expected classes before proceeding.

In [ ]:
# Section 1: Setup & Data Loading
# Purpose: Imports the required libraries, retrieves the supplied Kaggle dataset
# from Data_Source_URL, loads the selected CSV file, and performs the same
# initial inspection as the original notebook.

from pathlib import Path
from urllib.parse import urlparse, parse_qs
import io
import zipfile

import boto3
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

sns.set(style="whitegrid")

# -------------------------------------------------------------------------
# AWS SageMaker / S3 environment configuration
# Supplied in AWS_S3_Bucket_Credentials.txt.
#
# IMPORTANT: This notebook deliberately uses boto3 directly and does not
# depend on SageMaker SDK session-helper APIs. That makes the I/O compatible
# with SageMaker Python SDK v2 and v3. boto3 automatically uses
# the temporary IAM credentials attached to the SageMaker notebook/runtime.
# No static AWS access keys are embedded in this notebook.
# -------------------------------------------------------------------------
boto_session = boto3.Session()
region = boto_session.region_name

if not region:
    raise RuntimeError(
        "AWS region could not be resolved from the SageMaker environment. "
        "Set AWS_REGION/AWS_DEFAULT_REGION or configure the notebook session."
    )

s3_client = boto_session.client("s3")
sts_client = boto_session.client("sts")
caller_identity = sts_client.get_caller_identity()
caller_arn = caller_identity.get("Arn", "Unknown")

TEAM_ID = "team02"
STUDENT_ID = "s203"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/diabetes"

print(f"Bucket      : {BUCKET}")
print(f"Prefix      : {PREFIX}")
print(f"Region      : {region}")
print(f"AWS Caller  : {caller_arn}")
print(f"Team ID     : {TEAM_ID}")
print(f"Student ID  : {STUDENT_ID}")

# -------------------------------------------------------------------------
# Data source configuration
# Supplied in Data_Source_URL.txt.
#
# The supplied URL is a Kaggle dataset page, not a raw CSV URL. This code keeps
# that URL as the source of truth, extracts the dataset slug and selected file,
# downloads the public Kaggle dataset archive, and reads only the selected CSV.
# -------------------------------------------------------------------------
DATA_SOURCE_URL = (
    "https://www.kaggle.com/datasets/alexteboul/"
    "diabetes-health-indicators-dataset"
    "?select=diabetes_012_health_indicators_BRFSS2015.csv"
)

parsed_url = urlparse(DATA_SOURCE_URL)
path_parts = [part for part in parsed_url.path.split("/") if part]

if len(path_parts) < 3 or path_parts[0] != "datasets":
    raise ValueError(
        "DATA_SOURCE_URL must be a Kaggle dataset URL in the form "
        "https://www.kaggle.com/datasets/<owner>/<dataset>?select=<file.csv>"
    )

dataset_owner = path_parts[1]
dataset_name = path_parts[2]
selected_file = parse_qs(parsed_url.query).get("select", [None])[0]

if not selected_file:
    raise ValueError(
        "DATA_SOURCE_URL must include ?select=<csv-file-name> so the intended "
        "source file can be identified unambiguously."
    )

kaggle_download_url = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    f"{dataset_owner}/{dataset_name}"
)

print("\nData source:")
print("  Page URL      :", DATA_SOURCE_URL)
print("  Dataset       :", f"{dataset_owner}/{dataset_name}")
print("  Selected file :", selected_file)

response = requests.get(kaggle_download_url, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    archive_names = archive.namelist()

    # Prefer an exact archive member name; otherwise allow the same basename
    # inside a subdirectory.
    matching_members = [
        name
        for name in archive_names
        if name == selected_file or Path(name).name == selected_file
    ]

    if len(matching_members) != 1:
        csv_candidates = [
            name for name in archive_names if name.lower().endswith(".csv")
        ]
        raise FileNotFoundError(
            f"Expected exactly one archive member matching {selected_file!r}; "
            f"found {matching_members}. Available CSV files: {csv_candidates}"
        )

    source_member = matching_members[0]
    with archive.open(source_member) as csv_file:
        df = pd.read_csv(csv_file)

print("\nSource dataset loaded successfully from Data_Source_URL.")
print("Shape:", df.shape)
display(df.head())
print(df.dtypes)
print("Original target distribution (Diabetes_012):")
print(df["Diabetes_012"].value_counts())
print(df["Diabetes_012"].value_counts(normalize=True))

### Preserved console output from the report

```text
Bucket      : nyp-26s1-iti113
Prefix      : iti113/team02/data/diabetes
Region      : ap-southeast-1
AWS Caller  : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Team ID     : team02
Student ID  : s203

Data source:
  Page URL      : https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset?select=diabetes_012_health_indicators_BRFSS2015.csv
  Dataset       : alexteboul/diabetes-health-indicators-dataset
  Selected file : diabetes_012_health_indicators_BRFSS2015.csv

Source dataset loaded successfully from Data_Source_URL.
Shape: (253680, 22)

Original target distribution (Diabetes_012):
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

0.0    0.842412
2.0    0.139333
1.0    0.018255
Name: proportion, dtype: float64
```

`df.head()` displayed the first 5 rows of the 22-column dataset, and all source columns were reported as `float64`.

### Interpretation Comment - In [1]

The source dataset contains **253,680 rows and 22 columns**. The original `Diabetes_012` target is strongly imbalanced: **213,703 non-diabetes cases (84.24%)**, **4,631 prediabetes cases (1.83%)**, and **35,346 diabetes cases (13.93%)**.

The most notable finding is the very small prediabetes group. Diabetes cases are about **7.6 times** more common than prediabetes cases in the source label. This means the original three-class target contains substantially different class frequencies, which is important when interpreting any later binary transformation.


## 2. Convert to Binary Target

**Purpose:** Converts the original three-class diabetes outcome into a binary target suitable for binary classification.

**Binary class definition**
- `0` (**Negative**) = non-diabetes only (`Diabetes_012 = 0`)
- `1` (**Positive**) = prediabetes or diabetes (`Diabetes_012 = 1` or `2`)

**How to interpret this section:** The positive class represents anyone identified as having either prediabetes or diabetes, while the negative class contains only people without diabetes.

In [ ]:
# Section 2: Convert to Binary Target
# Purpose: Combines prediabetes and diabetes into the positive class,
# while keeping only non-diabetes in the negative class.

# Original Diabetes_012 coding:
#   0 = Non-diabetes
#   1 = Prediabetes
#   2 = Diabetes
#
# Binary mapping:
#   Diabetes_012 = 0      -> 0 (Negative: Non-diabetes)
#   Diabetes_012 = 1 or 2 -> 1 (Positive: Prediabetes or Diabetes)

expected_classes = {0, 1, 2}
observed_classes = set(df["Diabetes_012"].dropna().unique())

unexpected_classes = observed_classes - expected_classes
if unexpected_classes:
    raise ValueError(
        f"Unexpected values found in Diabetes_012: {sorted(unexpected_classes)}"
    )

df["Diabetes_binary"] = df["Diabetes_012"].isin([1, 2]).astype(int)

print("\nBinary target definition:")
print("  0 = Negative (Non-diabetes only)")
print("  1 = Positive (Prediabetes or Diabetes)")

print("\nBinary target distribution (Diabetes_binary):")
print(df["Diabetes_binary"].value_counts().sort_index())
print(df["Diabetes_binary"].value_counts(normalize=True).sort_index())

# Drop the original multiclass target from the binary-analysis dataset
df_binary = df.drop(columns=["Diabetes_012"])

### Preserved console output from the report

```text
Binary target definition:
  0 = Negative (Non-diabetes only)
  1 = Positive (Prediabetes or Diabetes)

Binary target distribution (Diabetes_binary):
Diabetes_binary
0    213703
1     39977
Name: count, dtype: int64

Diabetes_binary
0    0.842412
1    0.157588
Name: proportion, dtype: float64
```

### Interpretation Comment - In [2]

The derived `Diabetes_binary` target contains **213,703 negative cases (84.24%)** and **39,977 positive cases (15.76%)**. The positive class combines the **4,631 prediabetes** cases and **35,346 diabetes** cases.

The binary transformation therefore simplifies the prediction task to two classes, but it also removes the distinction between prediabetes and diabetes inside the positive class. The resulting target remains imbalanced, with the negative class accounting for more than four-fifths of the dataset.


## 3. Basic Data Quality Checks

**Purpose:** Checks missing values, summary statistics, and the number of unique values in each feature, then creates a cleaned working dataset.

**How to interpret this section:** Review missing-value counts and unusual ranges before relying on later plots or models.

In [ ]:
# Section 3: Basic Data Quality Checks
# Purpose: Checks missing values, summary statistics, and the number of unique values.

print("\nMissing values per column:")
missing_counts = df_binary.isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0])

print("\nDescriptive statistics:")
desc = df_binary.describe().T
display(desc)

print("\nNumber of unique values per feature:")
n_unique = df_binary.nunique().sort_values()
display(n_unique)

# Handle missing values if any (simple strategy: drop rows with NA)
df_binary_clean = df_binary.dropna()
print("\nShape after dropping rows with missing values:", df_binary_clean.shape)

### Preserved console output from the report

```text
Missing values per column:
Series([], dtype: int64)

Shape after dropping rows with missing values: (253680, 22)
```

Key descriptive ranges reported:

| Feature | Min | Max |
|---|---:|---:|
| BMI | 12 | 98 |
| MentHlth | 0 | 30 |
| PhysHlth | 0 | 30 |
| Age | 1 | 13 |
| Education | 1 | 6 |
| Income | 1 | 8 |

Number of unique values:

```text
HighBP                  2
HighChol                2
CholCheck               2
Smoker                  2
HeartDiseaseorAttack    2
Stroke                  2
PhysActivity            2
Fruits                  2
NoDocbcCost             2
Veggies                 2
HvyAlcoholConsump       2
AnyHealthcare           2
Sex                     2
Diabetes_binary         2
DiffWalk                2
GenHlth                 5
Education               6
Income                  8
Age                    13
MentHlth               31
PhysHlth               31
BMI                    84
dtype: int64
```

### Interpretation Comment - In [3]

No missing values were detected, and the dataset remains **253,680 rows × 22 columns** after the missing-value check. Therefore, no observations were removed because of missing data.

The reported ranges are consistent with the encoded BRFSS variables: BMI ranges from **12 to 98**, `MentHlth` and `PhysHlth` from **0 to 30**, and Age from **1 to 13**. The presence of large BMI values and high health-day counts indicates extreme observations, but this output alone does not establish that they are invalid records.


## 3A. Export Cleaned Binary Dataset

**Purpose:** Saves the cleaned binary dataset to a local CSV file and uploads the same cleaned artifact to the configured Amazon S3 bucket after removing missing rows and excluding the original three-class `Diabetes_012` column.

**Exported target definition**
- `0` (**Negative**) = non-diabetes only
- `1` (**Positive**) = prediabetes or diabetes

Excluding `Diabetes_012` prevents the original target from leaking into later binary-classification models.

In [ ]:
# Section 3A: Export Cleaned Binary Dataset
# Purpose: Saves the cleaned, model-ready binary dataset locally and uploads
# the same file to the configured Amazon S3 bucket for downstream SageMaker use.

from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_DATASET_PATH = OUTPUT_DIR / "diabetes_binary_cleaned_dataset.csv"

# Export predictors plus the revised binary target.
# Diabetes_012 has already been removed from df_binary_clean.
df_binary_clean.to_csv(CLEANED_DATASET_PATH, index=False)

# Validate the exported dataset definition.
if "Diabetes_012" in df_binary_clean.columns:
    raise ValueError(
        "Diabetes_012 must not appear in the cleaned binary export "
        "because it would duplicate/leak the target."
    )

exported_classes = set(df_binary_clean["Diabetes_binary"].dropna().unique())
if not exported_classes.issubset({0, 1}):
    raise ValueError(
        f"Unexpected Diabetes_binary values found: {sorted(exported_classes)}"
    )

print("Cleaned binary dataset exported locally successfully.")
print(f"Local file: {CLEANED_DATASET_PATH.resolve()}")
print(f"Rows: {df_binary_clean.shape[0]:,}")
print(f"Columns: {df_binary_clean.shape[1]:,}")

print("\nExported target distribution:")
print(
    df_binary_clean["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "Negative (Non-diabetes)",
            1: "Positive (Prediabetes or Diabetes)",
        }
    )
)

# -------------------------------------------------------------------------
# Upload the cleaned dataset to Amazon S3 using the IAM credentials
# automatically attached to the SageMaker notebook/runtime.
# -------------------------------------------------------------------------
S3_CLEANED_KEY = f"{PREFIX}/cleaned/{CLEANED_DATASET_PATH.name}"
S3_CLEANED_URI = f"s3://{BUCKET}/{S3_CLEANED_KEY}"

s3_client.upload_file(
    str(CLEANED_DATASET_PATH),
    BUCKET,
    S3_CLEANED_KEY,
)

# Confirm that the object exists and report its size.
s3_object = s3_client.head_object(Bucket=BUCKET, Key=S3_CLEANED_KEY)

print("\nCleaned binary dataset uploaded to Amazon S3 successfully.")
print(f"S3 URI: {S3_CLEANED_URI}")
print(f"S3 object size: {s3_object['ContentLength']:,} bytes")

### Preserved console output from the report

```text
Cleaned binary dataset exported locally successfully.
Local file: /home/sagemaker-user/EDA/outputs/diabetes_binary_cleaned_dataset.csv
Rows: 253,680
Columns: 22

Exported target distribution:
Diabetes_binary
Negative (Non-diabetes)                 213703
Positive (Prediabetes or Diabetes)       39977
Name: count, dtype: int64

Cleaned binary dataset uploaded to Amazon S3 successfully.
S3 URI: s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv
S3 object size: 22,230,794 bytes
```

### Interpretation Comment - In [4]

The cleaned binary dataset was exported successfully with **253,680 rows and 22 columns** and uploaded to the configured Amazon S3 location.

The exported modelling dataset contains the predictors and `Diabetes_binary` while excluding the original `Diabetes_012` field. This keeps the binary modelling target separate from the original three-class source label and avoids using that source label as a modelling predictor.


## 4. Class Balance Visualization

**Purpose:** Visualizes and prints the class distribution of the binary target.

**How to interpret this section:** Class `0` contains only non-diabetes cases. Class `1` combines prediabetes and diabetes cases. A large difference between class counts indicates class imbalance, which can affect model evaluation and training.

In [ ]:
# Section 4: Class Balance Visualization
# Purpose: Visualizes and prints the class distribution of the binary target.

target_col = "Diabetes_binary"

# Reindex to guarantee that the bars appear in 0, 1 order
class_counts = (
    df_binary_clean[target_col]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.xticks(
    [0, 1],
    ["Negative\n(Non-diabetes)", "Positive\n(Prediabetes or Diabetes)"],
    rotation=0,
)
plt.ylabel("Count")
plt.title("Binary Class Distribution")
plt.tight_layout()
plt.show()

print("\nClass counts:")
print(
    class_counts.rename(
        index={
            0: "Negative (Non-diabetes)",
            1: "Positive (Prediabetes or Diabetes)",
        }
    )
)

print("\nClass proportions:")
print(
    (class_counts / class_counts.sum()).rename(
        index={
            0: "Negative (Non-diabetes)",
            1: "Positive (Prediabetes or Diabetes)",
        }
    )
)

### Preserved console output from the report

```text
Class counts:
Diabetes_binary
Negative (Non-diabetes)                 213703
Positive (Prediabetes or Diabetes)       39977
Name: count, dtype: int64

Class proportions:
Diabetes_binary
Negative (Non-diabetes)                 0.842412
Positive (Prediabetes or Diabetes)      0.157588
Name: count, dtype: float64
```

Running the code cell regenerates the **Binary Class Distribution** bar chart.

### Interpretation Comment - In [5]

The class distribution is strongly imbalanced: **84.24% negative** and **15.76% positive**. The negative class is about **5.3 times** larger than the combined positive class.

This imbalance explains why overall accuracy by itself would not adequately describe model quality. A classifier could achieve a high accuracy while still failing to identify a large proportion of positive cases.


## 5. Identify Numeric vs Categorical-like Features

**Purpose:** Separates numeric features into categorical-like and continuous-ish groups based on the number of unique values.

**How to interpret this section:** The threshold of 10 unique values is a practical heuristic, not a strict statistical rule.

In [ ]:
# Section 5: Identify Numeric vs Categorical-like Features
# Purpose: Separates numeric features into categorical-like and continuous-ish groups.

feature_cols = [c for c in df_binary_clean.columns if c != target_col]
numeric_cols = (
    df_binary_clean[feature_cols]
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)

# Treat numeric columns with <= 10 unique values as categorical-like
cat_like_cols = [
    c for c in numeric_cols if df_binary_clean[c].nunique() <= 10
]
cont_cols = [c for c in numeric_cols if c not in cat_like_cols]

print("\nNumeric columns:", len(numeric_cols))
print(
    "Categorical-like numeric columns (<=10 unique values):",
    len(cat_like_cols),
)
print("Continuous-ish numeric columns:", len(cont_cols))
print("\nExample categorical-like columns:", cat_like_cols[:10])
print("Example continuous-ish columns:", cont_cols[:10])

### Preserved console output from the report

```text
Numeric columns: 21
Categorical-like numeric columns (<=10 unique values): 17
Continuous-ish numeric columns: 4

Example categorical-like columns:
['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
 'HvyAlcoholConsump']

Example continuous-ish columns:
['BMI', 'MentHlth', 'PhysHlth', 'Age']
```

### Interpretation Comment - In [6]

The unique-value heuristic classifies **17 variables as categorical-like numeric features** and **4 variables as continuous-ish features**: BMI, `MentHlth`, `PhysHlth`, and Age.

This result shows that storage type alone does not describe the statistical meaning of the predictors. Several variables are numerically encoded binary, ordinal, or categorical indicators. Age is a useful example: it has 13 ordered values, so it is ordinal even though the heuristic places it in the continuous-ish group.


## 6. Univariate Analysis: Histograms for Continuous-ish Features

**Purpose:** Plots histograms for up to 10 continuous-ish features to examine their distributions.

**How to interpret this section:** Look for skewness, extreme values, multiple peaks, and restricted ranges.

In [ ]:
# Section 6: Univariate Analysis: Histograms for Continuous-ish Features
# Purpose: Plots histograms for up to 10 continuous-ish features to examine their distributions.

if len(cont_cols) > 0:
    # Limit number of columns plotted at once if large
    cols_to_plot = cont_cols[:10]
    df_binary_clean[cols_to_plot].hist(figsize=(15, 10), bins=20)
    plt.suptitle("Histograms of continuous-ish features", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("\nNo continuous-ish columns identified (all appear categorical-like).")

### Preserved output from the report

Running the code cell regenerates histograms for:

- `BMI`
- `MentHlth`
- `PhysHlth`
- `Age`

### Interpretation Comment - In [7]

The preserved histogram results show that BMI is right-skewed with a long upper tail. `MentHlth` and `PhysHlth` are concentrated heavily at zero and extend to 30 days, while Age appears as a discrete 13-level ordinal variable.

These distributions are non-normal and contain extreme observations, especially for BMI and the health-day variables. The plots therefore show substantial skewness and discreteness rather than approximately Gaussian feature distributions.


## 7. Feature vs Target: Categorical-like Features

**Purpose:** Compares categorical-like features against the target using normalized stacked bar charts.

**How to interpret this section:** Each bar shows the proportion of negative cases and positive cases within a category.

In [ ]:
# Section 7: Feature vs Target: Categorical-like Features
# Purpose: Compares categorical-like features against the target using normalized stacked bar charts.

def plot_cat_vs_target(df, cat_col, target_col="Diabetes_binary"):
    ctab = pd.crosstab(
        df[cat_col],
        df[target_col],
        normalize="index",
    )

    # Ensure both binary classes are represented and consistently ordered
    ctab = ctab.reindex(columns=[0, 1], fill_value=0)

    ctab.plot(kind="bar", stacked=True, figsize=(6, 4))
    plt.title(f"{cat_col} vs {target_col}")
    plt.ylabel("Proportion")
    plt.legend(
        ["Negative (Non-diabetes)", "Positive (Prediabetes or Diabetes)"],
        title="Binary class",
    )
    plt.tight_layout()
    plt.show()

# Plot only a subset to keep things manageable
max_cat_plots = 12
print(
    f"\nPlotting up to {max_cat_plots} categorical-like columns vs target..."
)

for col in cat_like_cols[:max_cat_plots]:
    plot_cat_vs_target(df_binary_clean, col, target_col)

### Preserved output from the report

The report contains normalized stacked bar charts for the first 12 categorical-like predictors, including:

`HighBP`, `HighChol`, `CholCheck`, `Smoker`, `Stroke`, `HeartDiseaseorAttack`, `PhysActivity`, `Fruits`, `Veggies`, `HvyAlcoholConsump`, `AnyHealthcare`, and `NoDocbcCost`.

Running the code cell regenerates these charts.

### Interpretation Comment - In [8]

The normalized stacked bar charts show higher positive-class proportions for several risk-related categories, especially `HighBP`, `HighChol`, `Stroke`, and `HeartDiseaseorAttack`. Physical activity and some diet-related variables show smaller visible differences between the target classes.

These plots indicate associations between several categorical predictors and `Diabetes_binary`, but the visual differences do not establish causation. Features with modest individual separation may still contribute when considered together in a multivariable model.


## 8. Feature vs Target: Continuous-ish Features (Boxplots)

**Purpose:** Uses boxplots to compare continuous-ish features between the two target classes.

**How to interpret this section:** Differences in medians, spread, and outliers may indicate features that help distinguish the negative class from the positive class.

In [ ]:
# Section 8: Feature vs Target: Continuous-ish Features (Boxplots)
# Purpose: Uses boxplots to compare continuous-ish features between the two target classes.

max_cont_plots = 6
print(
    f"\nPlotting up to {max_cont_plots} continuous-ish columns vs target..."
)

for col in cont_cols[:max_cont_plots]:
    plt.figure(figsize=(6, 4))
    sns.boxplot(
        data=df_binary_clean,
        x=target_col,
        y=col,
        order=[0, 1],
    )
    plt.xticks(
        [0, 1],
        [
            "Negative\n(Non-diabetes)",
            "Positive\n(Prediabetes or Diabetes)",
        ],
    )
    plt.title(f"{col} by {target_col}")
    plt.tight_layout()
    plt.show()

### Preserved output from the report

Running the code cell regenerates the boxplots for `BMI`, `MentHlth`, `PhysHlth`, and `Age`.

### Interpretation Comment - In [9]

The preserved boxplot results show that the positive class has a higher BMI distribution, worse physical-health-day distribution, and an older age distribution. `MentHlth` also shows some difference between the two target classes.

The boxplots contain many apparent outliers, particularly for BMI and the health-day variables. These observations indicate wider and skewed feature distributions, but the plots alone do not show that the extreme values are data errors.


## 9. Correlation Analysis

**Purpose:** Calculates pairwise correlations, displays a heatmap, and ranks features by correlation with the target.

**How to interpret this section:** Correlation measures linear association only and does not imply causation.

In [ ]:
# Section 9: Correlation Analysis
# Purpose: Calculates pairwise correlations, displays a heatmap, and ranks features by target correlation.

corr = df_binary_clean.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation matrix (all numeric features)")
plt.tight_layout()
plt.show()

# Correlation with target
target_corr = corr[target_col].drop(target_col).sort_values(ascending=False)

print("\nTop positive correlations with Diabetes_binary:")
display(target_corr.head(15))

print("\nTop negative correlations with Diabetes_binary:")
display(target_corr.tail(15))

### Preserved console output from the report

Top positive correlations with `Diabetes_binary`:

```text
GenHlth                 0.300785
HighBP                  0.270334
BMI                     0.223851
DiffWalk                0.222155
HighChol                0.210290
Age                     0.185891
HeartDiseaseorAttack    0.176933
PhysHlth                0.174948
Stroke                  0.104800
MentHlth                0.074971
CholCheck               0.067879
Smoker                  0.062778
NoDocbcCost             0.038025
Sex                     0.029606
AnyHealthcare           0.014079
```

Strongest negative values reported:

```text
Fruits             -0.042088
HvyAlcoholConsump  -0.056682
Veggies            -0.059219
PhysActivity       -0.121392
Education          -0.131803
Income             -0.172794
```

Running the code cell regenerates the full correlation heatmap.

### Interpretation Comment - In [10]

The strongest positive correlations with `Diabetes_binary` are `GenHlth` (**0.301**), `HighBP` (**0.270**), BMI (**0.224**), `DiffWalk` (**0.222**), `HighChol` (**0.210**), Age (**0.186**), `HeartDiseaseorAttack` (**0.177**), and `PhysHlth` (**0.175**).

The strongest reported negative correlations are Income (**-0.173**), Education (**-0.132**), and `PhysActivity` (**-0.121**).

Overall, the relationships are weak to moderate rather than dominant. No single predictor has a correlation strong enough to explain the binary outcome by itself, which is consistent with diabetes risk being associated with multiple factors.


## 10. Multicollinearity Check (Highly Correlated Pairs)

**Purpose:** Identifies feature pairs with an absolute correlation above 0.80 as a simple multicollinearity check.

**How to interpret this section:** Highly correlated predictors may contain overlapping information and can affect some model types.

In [ ]:
# Section 10: Multicollinearity Check (Highly Correlated Pairs)
# Purpose: Identifies feature pairs with an absolute correlation above 0.80.

high_corr_pairs = []
cols = corr.columns.tolist()

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        col1, col2 = cols[i], cols[j]
        if abs(corr.loc[col1, col2]) > 0.8:  # threshold
            high_corr_pairs.append(
                (col1, col2, corr.loc[col1, col2])
            )

print("\nHighly correlated feature pairs (|r| > 0.8):")
for col1, col2, r in high_corr_pairs[:20]:
    print(f"{col1} - {col2}: {r:.3f}")

if not high_corr_pairs:
    print("None above the threshold.")

### Preserved console output from the report

```text
Highly correlated feature pairs (|r| > 0.8):
None above the threshold.
```

### Interpretation Comment - In [11]

No predictor pair has an absolute correlation above the selected **0.80** threshold.

The pairwise correlation screen therefore does not identify severe two-variable redundancy under this criterion. This result does not rule out moderate correlation or more complex multivariable collinearity, but it indicates that no pair is extremely correlated in the reported matrix.


## 11. Optional: Simple Baseline Model & Feature Importance

**Purpose:** Trains a simple Random Forest baseline, prints classification metrics, and displays feature importances.

**How to interpret this section:** This is an exploratory baseline rather than a fully tuned or production-ready model. The reported positive class combines prediabetes and diabetes, while the negative class contains only non-diabetes cases.

In [ ]:
# Section 11: Optional: Simple Baseline Model & Feature Importance
# Purpose: Trains a simple Random Forest baseline, prints classification metrics,
# and displays feature importances.

X = df_binary_clean.drop(columns=[target_col])
y = df_binary_clean[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("\nRandom Forest classification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=[
            "Negative (Non-diabetes)",
            "Positive (Prediabetes/Diabetes)",
        ],
        zero_division=0,
    )
)

feat_imp = pd.Series(
    rf.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)

print("\nTop 20 feature importances:")
display(feat_imp.head(20))

plt.figure(figsize=(8, 6))
feat_imp.head(20).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 20 Random Forest Feature Importances")
plt.tight_layout()
plt.show()

### Preserved console output from the report

```text
Random Forest classification report:

                                  precision    recall  f1-score   support
Negative (Non-diabetes)               0.87      0.96      0.91     42741
Positive (Prediabetes/Diabetes)       0.52      0.21      0.30      7995

accuracy                                                   0.84     50736
macro avg                              0.69      0.59      0.61     50736
weighted avg                           0.81      0.84      0.82     50736
```

Top 20 feature importances:

```text
BMI                     0.183736
Age                     0.122245
Income                  0.096980
PhysHlth                0.082164
GenHlth                 0.071218
Education               0.070617
MentHlth                0.064765
HighBP                  0.044836
Smoker                  0.032950
Fruits                  0.032894
HighChol                0.028534
Sex                     0.028133
Veggies                 0.026234
PhysActivity            0.025992
DiffWalk                0.023391
HeartDiseaseorAttack    0.017937
NoDocbcCost             0.014826
Stroke                  0.012052
AnyHealthcare           0.008579
HvyAlcoholConsump       0.007898
dtype: float64
```

Running the code cell regenerates the feature-importance chart.

### Interpretation Comment - In [12]

The exploratory Random Forest achieves about **84% overall accuracy**, but positive-class performance is much weaker: **precision 0.52, recall 0.21, and F1 0.30**. Negative-class recall is **0.96**.

The difference between overall accuracy and positive-class recall illustrates the effect of class imbalance. The model correctly classifies most negative cases but identifies only about one-fifth of positive cases.

The reported feature-importance ranking is led by BMI (**0.184**), Age (**0.122**), Income (**0.097**), `PhysHlth` (**0.082**), and `GenHlth` (**0.071**). This ranking broadly overlaps with variables that also showed visible class differences or stronger target correlations during the earlier EDA.


## 12. Short Text Summary Helper (Optional)

**Purpose:** Prints a short checklist that summarizes the main items to review after running the EDA.

**How to interpret this section:** Use the checklist to convert notebook outputs into written findings and conclusions while retaining the revised binary target definition.

In [ ]:
# Section 12: Short Text Summary Helper (Optional)
# Purpose: Prints a short checklist that summarizes the main items to review after the EDA.

print("\n--- EDA Summary Checklist ---")
print(
    "1. Binary target definition: 0 = Non-diabetes only; "
    "1 = Prediabetes or Diabetes."
)
print("2. Class balance: see the binary class plot and counts above.")
print(
    "3. Key predictors: see correlations and Random Forest feature importances."
)
print("4. Strong feature correlations: see high_corr_pairs above.")
print("5. Data issues: review missing values and binary class imbalance.")
print(
    "6. Interpretation: model results for class 1 refer to the combined "
    "Prediabetes/Diabetes positive class."
)

### Preserved console output from the report

```text
--- EDA Summary Checklist ---
1. Binary target definition: 0 = Non-diabetes only; 1 = Prediabetes or Diabetes.
2. Class balance: see the binary class plot and counts above.
3. Key predictors: see correlations and Random Forest feature importances.
4. Strong feature correlations: see high_corr_pairs above.
5. Data issues: review missing values and binary class imbalance.
6. Interpretation: model results for class 1 refer to the combined Prediabetes/Diabetes positive class.
```

### Final Interpretation Notes

After running all cells, summarize:

1. The number of observations and features used.
2. The revised binary target definition and class proportions.
3. Any missing values or unusual feature ranges.
4. The most noticeable feature distribution differences between the negative and positive classes.
5. Features with the strongest positive or negative target correlations.
6. Any highly correlated feature pairs.

# EDA Summary — Data Cleaning, Data Quality, Distribution, Relationships and Modelling Implications

## 1. Data Cleaning and Data-Quality Findings

The source dataset contains **253,680 rows and 22 columns**. The EDA did not identify missing values, so **no missing-value imputation and no row deletion were required** during the EDA cleaning stage.

The range checks also did not identify a documented invalid-value rule that required observations to be removed. BMI contains high values up to `98`, while `MentHlth` and `PhysHlth` extend from `0` to `30`. These values appear extreme in the distributions, but the executed EDA does not establish that they are data-entry errors. They were therefore retained rather than clipped or deleted.

The main data-cleaning transformation was the construction of the binary modelling label:

- `Diabetes_012 = 0` → `Diabetes_binary = 0`
- `Diabetes_012 = 1 or 2` → `Diabetes_binary = 1`

This produces a binary screening target while preserving the original three-class label as the analytical source-of-truth variable.

The modelling export keeps the predictors and `Diabetes_binary` while excluding `Diabetes_012` from the model input. This avoids direct target leakage from the original diagnosis-status label into the binary classifier.

## 2. Why the Original Three-Class Label Is Retained

The original `Diabetes_012` label is retained because it contains information that is lost when prediabetes and diabetes are merged into one positive class.

The original distribution is:

- non-diabetes: **213,703 (84.24%)**
- prediabetes: **4,631 (1.83%)**
- diabetes: **35,346 (13.93%)**

Prediabetes is therefore a very small group, and diabetes cases are about **7.6 times** more common than prediabetes cases.

Keeping `Diabetes_012` preserves the ability to:

- distinguish prediabetes from diabetes during descriptive analysis;
- verify exactly how the binary target was created;
- retain an auditable source label for future multiclass analysis;
- avoid losing clinically meaningful class information permanently.

For binary modelling, however, `Diabetes_012` is excluded from the predictor set because it is directly related to the derived target and would cause leakage.

## 3. Missing-Data Findings

The executed missing-value audit found **no missing values** in the dataset.

Therefore, the EDA cleaning stage did not perform:

- mean, median or mode imputation;
- model-based imputation;
- missing-category creation;
- row deletion caused by missingness.

This is an important result because it means the later preprocessing pipeline does not need a missing-value treatment step for this dataset unless new data received during deployment contains missing values.

## 4. Univariate Analysis Findings

The univariate analysis shows several different distribution types.

### BMI

BMI is continuous and visibly **right-skewed**, with a long upper tail and extreme high observations.

### MentHlth and PhysHlth

`MentHlth` and `PhysHlth` are highly concentrated at `0`, with values extending to `30`. Their distributions are therefore strongly non-normal and effectively **zero-inflated / highly right-skewed**.

### Age

Age is encoded using **13 ordered groups**. It is therefore an **ordinal variable**, not a truly continuous measurement, even though a simple unique-value heuristic may place it in a continuous-like group.

### Encoded categorical variables

Many other predictors are numeric in storage but represent binary, categorical or ordinal states. The EDA therefore correctly treats storage type and statistical meaning as separate concepts.

## 5. Skewness and How It Should Be Handled

The main skewed features identified by the EDA are:

- **BMI** — right-skewed;
- **MentHlth** — strongly right-skewed with many zero values;
- **PhysHlth** — strongly right-skewed with many zero values.

The EDA stage should **identify and document** the skewness rather than automatically alter the data.

Any transformation that changes model inputs belongs to the preprocessing stage and should be fitted using the training data only.

For the current Logistic Regression workflow, the most defensible preprocessing treatment is:

- retain valid observations rather than deleting extreme values only because they are visually unusual;
- standardize continuous / scale-sensitive predictors if scaling is introduced into the modelling pipeline;
- treat Age according to its ordinal meaning;
- apply log or power transformations only if later experiments show a measurable benefit and the transformation is compatible with the feature values.

No executed EDA result demonstrates that log transformation, winsorization, clipping or outlier deletion is required. These transformations should therefore not be described as completed cleaning steps.

## 6. Multivariate and Relationship Findings

The target-correlation analysis shows that no single feature has an overwhelmingly strong relationship with `Diabetes_binary`.

The strongest positive correlations reported are:

- `GenHlth`: approximately **0.301**
- `HighBP`: approximately **0.270**
- BMI: approximately **0.224**
- `DiffWalk`: approximately **0.222**
- `HighChol`: approximately **0.210**
- Age: approximately **0.186**
- `HeartDiseaseorAttack`: approximately **0.177**
- `PhysHlth`: approximately **0.175**

The stronger negative correlations include:

- Income: approximately **-0.173**
- Education: approximately **-0.132**
- `PhysActivity`: approximately **-0.121**

These are weak-to-moderate associations rather than dominant relationships. The result supports a multivariable modelling approach because the binary outcome is associated with several predictors rather than being explained by one variable alone.

The pairwise predictor-correlation screen did not identify any feature pair with absolute correlation above **0.80**. Therefore, the EDA does not show severe pairwise redundancy under that threshold. This does not prove that all multicollinearity is absent, but it does indicate that no two predictors are almost duplicates according to the reported correlation matrix.

## 7. Bivariate Class-Comparison Findings

The class-comparison plots show visible differences between negative and positive cases.

Higher positive-class proportions are especially apparent for variables such as:

- `HighBP`
- `HighChol`
- `Stroke`
- `HeartDiseaseorAttack`

The continuous / ordinal comparisons also show that the positive class tends to have:

- higher BMI;
- worse physical-health-day distributions;
- older Age groups.

These results are associations within the dataset and should not be interpreted as causal effects.

## 8. Feature-Importance Analysis

The notebook includes an **exploratory feature-importance exercise** as part of EDA. Its purpose is to check whether variables highlighted by descriptive analysis also appear influential in a multivariable predictive setting.

This exploratory analysis should be treated as supporting EDA evidence only. It is not the final feature-selection or model-selection method for Model A. The later Logistic Regression experiment provides model-specific explainability through fitted coefficients and scale-adjusted coefficient analysis.

Therefore, the EDA conclusion is that several variables appear informative, but the EDA does **not** justify removing the remaining predictors solely on the basis of exploratory importance ranking.

## 9. Class-Imbalance Finding

The derived binary target contains:

- negative class: **213,703 (84.24%)**
- positive class: **39,977 (15.76%)**

The dataset is therefore substantially imbalanced.

This explains why overall accuracy is not sufficient as the main evaluation metric. A model can obtain high accuracy by predicting the majority class well while still missing many positive cases.

The EDA stage should identify and quantify this imbalance. The actual imbalance treatment belongs to the modelling / preprocessing workflow.

The later Logistic Regression experiment shows that `class_weight="balanced"` materially changes minority-class detection. In the final modelling workflow, class balancing is therefore handled through the model's class-weight setting rather than through row deletion or synthetic data generation during EDA.

## 10. EDA Stage Versus Preprocessing Stage

| Activity | EDA / Data-Quality Stage | Preprocessing / Modelling Stage |
|---|---|---|
| Inspect shape, schema and data types | Yes | No |
| Audit missing values | Yes | Apply treatment only if required |
| Check value ranges and unusual observations | Yes | Transform/remove only if justified |
| Quantify class imbalance | Yes | Apply class weighting / resampling strategy |
| Plot univariate distributions | Yes | No |
| Identify skewness | Yes | Apply transformations only if experimentally justified |
| Examine class-wise distributions | Yes | No |
| Correlation and relationship analysis | Yes | No |
| Identify potential leakage | Yes | Enforce feature exclusion in pipeline |
| Preserve `Diabetes_012` as source label | Yes | Exclude it from model features |
| Derive `Diabetes_binary` | Yes, as target-definition step | Consume as modelling target |
| Train/test split | No | Yes |
| Standardization / scaling | No | Yes, if used |
| Encoding required by model pipeline | No | Yes |
| Class-weight configuration | No | Yes |
| Hyperparameter tuning | No | Yes |
| Threshold optimisation | No | Yes |
| Final model evaluation | No | Yes |

## 11. Overall EDA Conclusion

The EDA shows that the dataset is structurally clean in terms of missingness and does not require row removal or imputation. The main data-quality considerations are class imbalance, skewed numeric distributions, ordinal/categorical encoding, possible extreme but plausible observations, and prevention of target leakage.

The binary modelling target is suitable for the screening objective, but retaining the original `Diabetes_012` variable remains important because it preserves the distinction between non-diabetes, prediabetes and diabetes for analysis and auditability.

The strongest EDA signal is not a single dominant predictor. Instead, several health-status, cardiovascular-risk, BMI, mobility, age and socioeconomic variables show weak-to-moderate relationships with the binary target. This supports the use of a multivariable classifier.

The EDA therefore performs **data understanding, quality assessment, anomaly identification, label definition and relationship analysis**, while **scaling, transformation, imbalance treatment, train/test splitting, model fitting, hyperparameter tuning and threshold optimisation** remain part of the downstream preprocessing and modelling workflow.


### Interpretation Comment - In [13]

The final helper cell does not perform a new statistical analysis. It prints a checklist that summarizes the main EDA outputs: the binary target definition, class imbalance, key predictors, correlation results, highly correlated feature pairs, and interpretation of the combined positive class.

The checklist confirms that the notebook has covered the main descriptive issues needed to understand the dataset before modelling. In particular, it reinforces that model results for class `1` refer to the combined **prediabetes-or-diabetes** group rather than to prediabetes alone.


# Overall EDA Review, Conclusion & Recommendation

## Overall review

- The EDA execution is complete and reproducible. The source data contains **253,680 records**, no missing values were detected, and the cleaned binary modelling artifact was exported successfully to S3.
- The most important data-governance finding is the original target structure: prediabetes is only **4,631 records (1.83%)** versus **35,346 diabetes records (13.93%)**. Retaining `Diabetes_012` preserves this visibility; `Diabetes_binary` should be a derived modelling label, not a replacement for the original EDA label.
- The binary modelling target is imbalanced at **84.24% negative versus 15.76% positive**. This explains why overall accuracy can be misleading.

## Conclusion

- The EDA identifies useful signal in general health, high blood pressure, BMI, walking difficulty, cholesterol, age, heart disease, and physical health.
- No severe pairwise multicollinearity was found at `|r| > 0.80`.
- The exploratory Random Forest demonstrates the main modelling risk: despite about 84% accuracy, positive-class recall is only 0.21. A model selected on accuracy alone would therefore be unsuitable for minority-class detection.

## Recommendation for the following stage

- Proceed to MLflow-managed model experimentation using a leakage-safe, stratified workflow. Keep the held-out test set untouched until model and hyperparameter selection are complete.
- Use PR-AUC as the primary selection metric and report ROC-AUC, recall, F1, balanced accuracy, and precision.
- Compare unweighted and class-balanced Logistic Regression settings and review the classification threshold after hyperparameter selection.
- Track the data profile, feature schema, preprocessing choices, parameters, CV metrics, final test metrics, and model artifacts in MLflow.
- Before production promotion, rerun the selected configuration on the full dataset and verify that minority-class performance remains stable.